# Bài 04 · Thực hành và lời giải
Đọc bảng bốn điểm, tự tính trước rồi chạy. Dữ liệu thực dùng cùng train/test và quy tắc như bài giảng.
Các ô mã có assert để kiểm tra kết quả; notebook chứa lời giải nên dùng sau lượt làm độc lập.


In [1]:
"""Bài thực hành có lời giải, dùng cùng dữ liệu và quy tắc với slide.
Chạy từ thư mục ML_redesign_v1: python lab_bai04.py
"""


## Thiết lập


In [2]:
from pathlib import Path
import sys, json
if hasattr(sys.stdout,"reconfigure"):sys.stdout.reconfigure(encoding="utf-8")
ROOT=Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
local_deps=ROOT/'.build/pydeps'
if not local_deps.exists():local_deps=ROOT.parent/'.build/pydeps'
if local_deps.exists():sys.path.insert(0,str(local_deps))
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,root_mean_squared_error



## A. Ví dụ mô phỏng: tự tính trước khi chạy


In [3]:
x=np.array([1.,2.,3.,4.])  # đơn vị 100 km
y=np.array([3.,4.,6.,7.])  # ngày
z=x-x.mean()
w=0.;eta=.4
error=5+w*z-y
gradient=np.mean(z*error)
w_new=w-eta*gradient
loss_before=np.mean(error**2)/2
loss_after=np.mean((5+w_new*z-y)**2)/2
print('VÍ DỤ MÔ PHỎNG:',{'gradient':gradient,'w_1':w_new,'J_0':loss_before,'J_1':loss_after})
assert np.isclose(w_new,.7) and np.isclose(loss_after,.33125)



VÍ DỤ MÔ PHỎNG: {'gradient': np.float64(-1.75), 'w_1': np.float64(0.7000000000000001), 'J_0': np.float64(1.25), 'J_1': np.float64(0.3312500000000002)}


## B. Dữ liệu thật: bảng đã chuẩn bị ở cấp đơn hàng


In [4]:
d=pd.read_csv(ROOT/'data/prepared/olist_model_table.csv')
train=d[d['split']=='train'].copy()
test=d[d['split']=='test'].copy()
print('Số đơn train/test:',len(train),len(test))



Số đơn train/test: 29031 40266


## C. Chỉ fit các phép biến đổi trên train


In [5]:
imputer=SimpleImputer(strategy='median')
X_train=imputer.fit_transform(train[['distance_km']])
X_test=imputer.transform(test[['distance_km']])
scaler=StandardScaler()
x_train=scaler.fit_transform(X_train).ravel()
x_test=scaler.transform(X_test).ravel()
y_train=train['delivery_days'].to_numpy()
y_test=test['delivery_days'].to_numpy()



## D. Học đồng thời b và w bằng GD


In [6]:
b=0.;w=0.;eta=.1
history=[]
for step in range(100):
    prediction=b+w*x_train
    error=prediction-y_train
    grad_w=np.mean(x_train*error)
    grad_b=np.mean(error)
    history.append({'step':step,'loss':float(np.mean(error**2)/2)})
    w=w-eta*grad_w
    b=b-eta*grad_b



## E. Đối chiếu triển khai và đánh giá trên test


In [7]:
ols=LinearRegression().fit(x_train.reshape(-1,1),y_train)
assert abs(b-ols.intercept_)<.001
assert abs(w-ols.coef_[0])<.001
pred_gd=b+w*x_test
pred_ols=ols.predict(x_test.reshape(-1,1))
pred_baseline=np.full(len(test),y_train.mean())
rows=[]
for name,pred in [('baseline_mean_train',pred_baseline),('GD_100_steps',pred_gd),('least_squares',pred_ols)]:
    rows.append({'model':name,'MAE_days':mean_absolute_error(y_test,pred),'RMSE_days':root_mean_squared_error(y_test,pred)})
print(pd.DataFrame(rows).to_string(index=False))
reference=json.loads((ROOT/'results/olist_pilot.json').read_text(encoding='utf-8'))
expected=next(t for t in reference['regression'] if t['name']=='distance')['test']
assert np.isclose(rows[-1]['MAE_days'],expected['mae'],atol=1e-9)
assert np.isclose(rows[-1]['RMSE_days'],expected['rmse'],atol=1e-9)



              model  MAE_days  RMSE_days
baseline_mean_train  6.851264  10.167486
       GD_100_steps  5.939625   9.351598
      least_squares  5.939645   9.351536


## F. Đổi tham số về hệ đơn vị km để diễn giải


In [8]:
w_km=w/scaler.scale_[0]
b_km=b-w*scaler.mean_[0]/scaler.scale_[0]
print('Mô hình GD gần đúng:',{'b_days':b_km,'w_days_per_km':w_km})
print('Dự đoán cho 400 km:',b_km+w_km*400,'ngày')



Mô hình GD gần đúng: {'b_days': np.float64(7.9079817324931065), 'w_days_per_km': np.float64(0.005638998750298331)}
Dự đoán cho 400 km: 10.16358123261244 ngày


## G. Câu hỏi sau thực hành


In [9]:
# 1. Dự đoán cho 400 km là một cam kết chính xác hay một ước lượng? Vì sao?
# 2. Vì sao không fit StandardScaler trên cả train và test?
# 3. Nếu dùng ngày giao thực tế làm đầu vào thì điểm số còn ý nghĩa gì?
# 4. Thử eta khác trên train. Giữ nguyên test, không chọn eta bằng điểm test.
out=ROOT/'results/lab_check.json'
out.write_text(json.dumps({'metrics':rows,'b_days':b_km,'w_days_per_km':w_km,'implementation_matches_least_squares':True},indent=2),encoding='utf-8')


## H. Đối chiếu hình mới
1. Tại (w,b) = (0,5;1), tính đồng thời hai gradient với η = 0,05.
2. Vì sao hai bộ sai số có cùng MAE nhưng RMSE khác?


In [10]:
toy_x = np.array([1., 2., 3., 4.])
toy_y = np.array([3., 4., 6., 7.])
design = np.column_stack([toy_x, np.ones(4)])  # thứ tự tham số: w, b
theta = np.array([0.5, 1.0])
error = design @ theta - toy_y
g = design.T @ error / len(toy_y)
updated = theta - 0.05 * g
assert np.allclose(g, [-8, -2.75])
assert np.allclose(updated, [0.9, 1.1375])
print('Gradient, tham số mới:', g, updated)
for errors in [np.array([2, 2, 2, 2]), np.array([0, 0, 0, 8])]:
    print('MAE, RMSE:', np.mean(np.abs(errors)), np.sqrt(np.mean(errors**2)))


Gradient, tham số mới: [-8.   -2.75] [0.9    1.1375]
MAE, RMSE: 2.0 2.0
MAE, RMSE: 2.0 4.0
